In [7]:
#!pip install -q torch transformers scikit-learn pandas matplotlib tqdm

import os
import copy
import random
import contextlib

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import transformers
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

Torch version: 2.11.0+cu128
Transformers version: 5.14.1


In [8]:
# ===== Konfigurasi utama =====
MODEL_NAMES = ["roberta-base"]              # model yang akan dilatih & di-ensemble (lihat bagian 9 untuk ide model lain)
LOCAL_DIRS = {                              # (opsional) folder lokal format Hugging Face untuk tiap model
    "roberta-base": "../pretrained/roberta_base",   # jika folder ini valid → dipakai; jika tidak → unduh dari Hub
}

TEXT = "comment_text"

SEEDS = [42]                                # daftar seed; tiap seed = satu kali K-Fold lengkap
N_FOLDS = 5                                 # jumlah lipatan cross-validation
EPOCHS = 3                                  # epoch per fold
BATCH_SIZE = 16                             # ukuran batch train
LR = 2e-5                                   # learning rate puncak
WEIGHT_DECAY = 0.01                         # regularisasi L2 (tidak untuk bias & LayerNorm)
WARMUP_RATIO = 0.1                          # 10% langkah awal untuk warmup
MAX_LEN = 96                                # batas token per input (keyword + tweet)
LABEL_SMOOTHING = 0.1                       # ε pada label smoothing
DEBUG = False                               # True → versi mini untuk tes pipeline

if DEBUG:                                   # mode tes cepat
    N_FOLDS, EPOCHS = 2, 1                  #   cukup 2 fold & 1 epoch
    print("⚠️ DEBUG aktif: hasil TIDAK untuk disubmit.")   # peringatan


def set_seed(seed):                         # kunci semua sumber keacakan
    random.seed(seed)                       #   Python
    np.random.seed(seed)                    #   NumPy
    torch.manual_seed(seed)                 #   PyTorch CPU
    torch.cuda.manual_seed_all(seed)        #   PyTorch GPU (aman walau tanpa GPU)

if torch.cuda.is_available():               # GPU NVIDIA tersedia?
    device = torch.device("cuda")           #   pakai GPU
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():   # Apple GPU?
    device = torch.device("mps")            #   pakai MPS
else:                                       # selain itu
    device = torch.device("cpu")            #   CPU (sangat lambat untuk notebook ini)
use_amp = device.type == "cuda"             # fp16 hanya di GPU NVIDIA
print("Perangkat:", device, "| AMP:", use_amp) # tampilkan

Perangkat: cuda | AMP: True


In [9]:
KAGGLE_DIR = "/kaggle/input/nlp-getting-started"     # lokasi data di Kaggle

def find_file(filename):                              # cari file di beberapa folder kandidat
    for folder in [".", KAGGLE_DIR]:                  #   coba folder kerja, lalu folder Kaggle
        path = os.path.join(folder, filename)         #     bentuk path lengkap
        if os.path.exists(path):                      #     ada?
            return path                               #       kembalikan
    raise FileNotFoundError(f"{filename} tidak ditemukan")   # tidak ada di mana pun

df_train = pd.read_csv(find_file("train.csv"))        # data latih
df_test = pd.read_csv(find_file("test.csv"))          # data uji

if DEBUG:                                             # mode tes cepat → kecilkan data
    df_train = df_train.sample(400, random_state=0).reset_index(drop=True)   # 400 baris acak
    df_test = df_test.head(200).reset_index(drop=True)                       # 200 baris pertama

print("Latih:", df_train.shape, "| Uji:", df_test.shape)   # cek ukuran

Latih: (159571, 8) | Uji: (153164, 2)


In [10]:
df_train.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [11]:
df_test.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [12]:
def normalize_tweet(text):                                    # fungsi pembersih teks minimal
    text = html.unescape(str(text))                           #   &amp; → &  ; str() menjaga agar NaN tak membuat error
    text = re.sub(r"https?://\S+|www\.\S+", "http", text)     #   semua URL (http/https/www) → kata "http"
    text = re.sub(r"@\w+", "@user", text)                     #   semua mention @nama → "@user"
    text = re.sub(r"\s+", " ", text).strip()                  #   rapikan spasi berlebih & spasi di tepi
    return text                                               #   hasil bersih


# --- Lihat dulu contoh normalisasi ---
for t in df_train["text"].head(3):                            # ambil 3 tweet pertama
    print("ASLI :", t)                                        #   sebelum
    print("BERSIH:", normalize_tweet(t), "\n")                #   sesudah

KeyError: 'text'

In [ ]:
df_train["text_norm"] = df_train["text"].map(normalize_tweet)     # teks ternormalisasi untuk train
df_test["text_norm"] = df_test["text"].map(normalize_tweet)       # idem untuk test

grp = df_train.groupby("text_norm")["target"]                     # kelompokkan baris berdasarkan teks yang sama
df_train["dup_n"] = grp.transform("size")                         # berapa kali teks itu muncul
df_train["dup_mean"] = grp.transform("mean")                      # rata-rata label di kelompok (0=semua 0, 1=semua 1, di antaranya=campur)

conflict = (df_train["dup_n"] > 1) & (df_train["dup_mean"] > 0) & (df_train["dup_mean"] < 1)   # duplikat DENGAN label berbeda
tie = conflict & (df_train["dup_mean"] == 0.5)                    # sub-kasus seri 50:50

print(f"Baris dengan label bertentangan: {conflict.sum()} | di antaranya seri: {tie.sum()}")   # statistik
df_train.loc[conflict, ["text", "target"]].sort_values("text").head(10)   # tampilkan contoh pertentangan

In [ ]:
n_before = len(df_train)                                          # jumlah baris sebelum dibersihkan

df_train.loc[conflict, "target"] = (df_train.loc[conflict, "dup_mean"] > 0.5).astype(int)   # seragamkan ke label mayoritas
df_train = df_train[~tie]                                         # buang kelompok yang seri (ambigu)
df_train = df_train.drop_duplicates(subset="text_norm", keep="first").reset_index(drop=True)   # sisakan 1 baris per teks
df_train["target"] = df_train["target"].astype(int)               # pastikan label bertipe int (bukan float)

print(f"Baris data latih: {n_before} → {len(df_train)}")               # berapa baris yang hilang
print("Proporsi kelas:", df_train["target"].value_counts(normalize=True).round(3).to_dict())   # sebaran kelas setelah bersih